In [8]:
#create env using conda commands:conda_dicom2omop_env
!pip list

#add tqdm
!pip install tqdm

Package           Version
----------------- ------------
asttokens         3.0.0
colorama          0.4.6
comm              0.2.3
contourpy         1.3.2
cycler            0.12.1
debugpy           1.8.16
decorator         5.2.1
exceptiongroup    1.3.0
executing         2.2.0
fonttools         4.62.0
ipykernel         6.30.1
ipython           8.37.0
jedi              0.19.2
jupyter_client    8.6.3
jupyter_core      5.8.1
kiwisolver        1.5.0
matplotlib        3.10.8
matplotlib-inline 0.1.7
munkres           1.1.4
nest-asyncio      1.6.0
numpy             2.2.6
packaging         25.0
pandas            2.3.3
parso             0.8.5
pickleshare       0.7.5
pillow            12.1.1
pip               26.0.1
platformdirs      4.4.0
prompt_toolkit    3.0.51
psutil            7.0.0
pure_eval         0.2.3
pyarrow           23.0.1
pydicom           3.0.1
Pygments          2.19.2
pyparsing         3.3.2
PySide6           6.10.2
python-dateutil   2.9.0.post0
pytz              2026.1.post1
pywin3

In [ ]:
#basic EDA of sample of ToothFairy dataset (4 folders of .dcm DICOM files - P1, P2, P3, P4)
# ============================================================
#step 1 - imports and path
# ============================================================
import os
from pathlib import Path
import pydicom
import pandas as pd
import numpy as np
from tqdm import tqdm
import time

#subset of 4 folders only here!
DATA_DIR = Path(r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy")
print('data directory', DATA_DIR)
print('exists', DATA_DIR.exists())

all_files = []
non_dcm_files = []
dcm_files = []

# ============================================================
#step 2 - count files and look at folrder structure
# ============================================================
for root, dir, files in os.walk(DATA_DIR):   #root - folder, dirs - subfolder, files - files 
    print(root)  #root is folder + P1, P2, ...
    for file in files:
        full_path = Path(root) / file
        all_files.append(full_path)
        if file.lower().endswith('.dcm'):
            dcm_files.append(full_path)
        if not file.lower().endswith('.dcm'):
            non_dcm_files.append(full_path)

print('Total files:',len(all_files))
print('total dcm files:', len(dcm_files))
print('\nfirst 5 DICOM files:')
for file in dcm_files[:5]:
    print(file)
print('-'*20)
print('all non dcm files')
for file in non_dcm_files:
    print(file)

data directory C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy
exists True
C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy
C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P1
C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P2
C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P3
C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P4
Total files: 1166
total dcm files: 1162

first 5 DICOM files:
C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P1\55aci73eg45102001000.dcm
C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P1\55aci73eg45102001001.dcm
C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P1\55aci73eg45103011000.dcm
C:\Users\julie\AI agent

In [2]:
#open a DICOMDIR file
dicomdir_path = r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P1\DICOMDIR"
ds = pydicom.dcmread(dicomdir_path)
print(ds)



Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 162
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: Media Storage Directory Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.76.380.18.10.1131014132236753.30299
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.76.380.18.10
(0002,0013) Implementation Version Name         SH: '8_0'
(0002,0016) Source Application Entity Title     AE: 'CT01'
-------------------------------------------------
(0004,0000) Group Length                        UL: 86524
(0004,1130) File-set ID                         CS: 'SEQ_ASSIALI'
(0004,1200) Offset of the First Directory Recor UL: 384
(0004,1202) Offset of the Last Directory Record UL: 384
(0004,1212) File-set Consistency Flag           US: 0
(0004,1220)  Directory Record Sequence  339 item(s) --

In [3]:
for record in ds.DirectoryRecordSequence[:10]:
    print(record.DirectoryRecordType)
    print(record)
    print('-'*40)

#we see patient, study, series, image

#DICOM data hierarchy: PatientID, StudyInstanceUID, SeriesInstanceUID
#study - one imaging session/visit like one cbct on date, series - one set of images in study - axial slides, different protocols, etc

PATIENT
(0004,1400) Offset of the Next Directory Record UL: 0
(0004,1410) Record In-use Flag                  US: 65535
(0004,1420) Offset of Referenced Lower-Level Di UL: 524
(0004,1430) Directory Record Type               CS: 'PATIENT'
(0008,0005) Specific Character Set              CS: 'ISO_IR 192'
(0010,0010) Patient's Name                      PN: 'prova1'
(0010,0020) Patient ID                          LO: '00554076'
(0010,0030) Patient's Birth Date                DA: '19831017'
(0010,0040) Patient's Sex                       CS: 'F'
----------------------------------------
STUDY
(0004,1400) Offset of the Next Directory Record UL: 0
(0004,1410) Record In-use Flag                  US: 65535
(0004,1420) Offset of Referenced Lower-Level Di UL: 760
(0004,1430) Directory Record Type               CS: 'STUDY'
(0008,0005) Specific Character Set              CS: 'ISO_IR 192'
(0008,0020) Study Date                          DA: '20200918'
(0008,0030) Study Time                          TM:

In [4]:
# ============================================================
#step 3 - open a single dicom file to look at header metadata
# ============================================================
sample_dcm = dcm_files[0]
print('sample dcm:', sample_dcm)
ds_sample = pydicom.dcmread(sample_dcm, stop_before_pixels=True)  #arg says do not laod image pixel array
print(ds_sample)

sample dcm: C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P1\55aci73eg45102001000.dcm
Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 172
(0002,0001) File Meta Information Version       OB: b'\x00\x01'
(0002,0002) Media Storage SOP Class UID         UI: CT Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.76.380.18.10.1131014132236753.30299.91.1
(0002,0010) Transfer Syntax UID                 UI: Explicit VR Little Endian
(0002,0012) Implementation Class UID            UI: 1.76.380.18.10
(0002,0013) Implementation Version Name         SH: '8_0'
(0002,0016) Source Application Entity Title     AE: 'CT01'
-------------------------------------------------
(0008,0000) Group Length                        UL: 418
(0008,0005) Specific Character Set              CS: 'ISO_IR 192'
(0008,0008) Image Type                          CS: ['ORIGINAL', 'PRIMARY', 'LOCALIZER']
(0008,0012) Inst

In [5]:
#show DICOM tags in alphabetical order by keyword, to verify important parameters are 
#DICOM element made of : (Tag) Name   VR: Value   (0010,0020) Patient ID   LO: '00554076'
#Tag (0010,0020)      ID
#Name: Patient ID     human readable label
# Keyword: PatientID  python code friendly version 
#value: '00554076'   actual data 
elements = []
for elem in ds_sample:
    elements.append({
        "keyword": elem.keyword,
        "name": elem.name,
        "tag": str(elem.tag),
        "value":str(elem.value)[:100]  #truncate super long values
    })
df_tags = pd.DataFrame(elements)

#sort by keyword alphabetically
df_tags = df_tags.sort_values('keyword')

#see all 144 rows in notebook
pd.set_option('display.max_rows', None)
df_tags


,keyword,name,tag,value
0,,Group Length,"(0008,0000)",418
81,,Private tag data,"(0019,1041)",VG12004S
80,,Private tag data,"(0019,1040)",0
79,,Private tag data,"(0019,1039)",1
78,,Private tag data,"(0019,1038)",Boosted dose
77,,Private tag data,"(0019,1037)",
76,,Private tag data,"(0019,1036)",0
75,,Private tag data,"(0019,1033)","[2020, 09, 18, 09, 48, 55, 180, 05]"
74,,Private tag data,"(0019,1032)",15
73,,Private tag data,"(0019,1031)",13


In [ ]:
# ============================================================
#step 3 - DICOM TAG EXTRACTION
# ============================================================

pd.reset_option('display.max_rows')  #reset to normal 

#important tags to need for OMOP, NOT complete so need to add and substract as we go
#since OMOP Image_occurance table maps at Series not instance level, can extract from one file per series for speed
# to avoid duplicate info

#one per series variable - keep data for only one DICOM file per series, for speed
ONE_PER_SERIES = True

DATA_DIR = Path(r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy")

STRING_TAGS = [
    'StudyInstanceUID', 
    'SeriesInstanceUID', 
    'SOPInstanceUID', 
    'SOPClassUID',
    'PatientID', 
    'StudyDate', 
    'StudyDescription', 
    'Modality', 
    'BodyPartExamined', 
    'PatientPosition', 
    'Manufacturer',
    'ManufacturerModelName', 
    'SoftwareVersions', 
    'InstitutionName',
]
NUMERIC_TAGS = [
    'SeriesNumber',
    'KVP', 
    'XRayTubeCurrent', 
    'ExposureTime', 
    'SliceThickness',
    'Rows',
    'Columns', 
    'BitsAllocated', 
    'NumberOfFrames'
]

#calculate number of dcm files in dataset
all_files = []
for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith('.dcm'):
            file_path = os.path.join(root, file)
            all_files.append(file_path)


print('scanning DICOM files')
metadata_records = []
seen_series = set() 
failed_files = []

for file_path in tqdm(all_files, desc='Extracting DICOM Metadata', unit='file'):
        try:
            ds = pydicom.dcmread(file_path, stop_before_pixels=True)

            #avoid duplication of images in same series
            series_uid = str(ds.get('SeriesInstanceUID', 'MISSING'))
            if ONE_PER_SERIES and series_uid in seen_series:
                continue
            seen_series.add(series_uid)

            #get patient folder info from current filepath - top of any possible nested folders in DATA_DIR
            file_dir = os.path.dirname(file_path)
            rel_path = os.path.relpath(file_dir, DATA_DIR)
            patient_folder = rel_path.split(os.sep)[0]
            record = {
                #file info
                'FilePath': file_path,
                'PatientFolder': patient_folder,
            }

            #extract String Fields - str or 'MISSING'
            #extract Numeric Fields - float or None
            for tag in STRING_TAGS:
                val = ds.get(tag, None)
                record[tag] = str(val).strip() if val else 'MISSING'
            for tag in NUMERIC_TAGS:
                val = ds.get(tag, None)
                try:
                    record[tag] = float(val) if val is not None else np.nan  #missing tag
                except:
                    record[tag] = np.nan     #avoid float "dfdd 132" crashing system

            #PixelSpacing calculations
            pixel_spacing = ds.get('PixelSpacing', None)
            ps_row = float(pixel_spacing[0]) if pixel_spacing else None
            ps_col = float(pixel_spacing[1]) if pixel_spacing else None
            record['PixelSpacing_Row'] = ps_row
            record['PixelSpacing_Col'] = ps_col

            metadata_records.append(record)
        
        except Exception as e:
            failed_files.append({'FilePath': file_path, 'Error': str(e)})
    
print('all done!')
df = pd.DataFrame(metadata_records)




scanning DICOM files


Extracting DICOM Metadata: 100%|██████████| 1162/1162 [00:00<00:00, 1421.29file/s]

all done!


In [31]:

DATA_DIR = Path(r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy")

print("=== Full Directory Structure ===")
all_patient_folders = set()

for root, dirs, files in os.walk(DATA_DIR):
    dcm_files = [f for f in files if f.lower().endswith('.dcm')]
    if dcm_files:
        rel = os.path.relpath(root, DATA_DIR)
        parts = rel.split(os.sep)
        patient_folder = parts[0]
        all_patient_folders.add(patient_folder)
        print(f"  root          : {root}")
        print(f"  rel_path      : {rel}")
        print(f"  parts         : {parts}")
        print(f"  patient_folder: {patient_folder}")
        print(f"  dcm count     : {len(dcm_files)}")
        print()

print(f"Unique patient_folder values found: {all_patient_folders}")
print(f"Count: {len(all_patient_folders)}")

=== Full Directory Structure ===
  root          : C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P1
  rel_path      : P1
  parts         : ['P1']
  patient_folder: P1
  dcm count     : 332

  root          : C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P2
  rel_path      : P2
  parts         : ['P2']
  patient_folder: P2
  dcm count     : 279

  root          : C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P3
  rel_path      : P3
  parts         : ['P3']
  patient_folder: P3
  dcm count     : 273

  root          : C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy\P4
  rel_path      : P4
  parts         : ['P4']
  patient_folder: P4
  dcm count     : 278

Unique patient_folder values found: {'P2', 'P4', 'P1', 'P3'}
Count: 4


In [33]:
# ============================================================
#ToothFairy Dataset analysis
# ============================================================
#high level summary
print(f"patient folders: {df['PatientFolder'].nunique()}")
print(f"unique studies: {df['StudyInstanceUID'].nunique()}")
print(f"unique series: {df['SeriesInstanceUID'].nunique()}")
print(f"Modalities found: {df['Modality'].unique().tolist()}")
print(f"BodyPartExamined: {df['BodyPartExamined'].unique().tolist()}")
print(f"Manufacturers: {df['Manufacturer'].unique().tolist()}")
print(f"Scanner models: {df['ManufacturerModelName'].unique().tolist()}")

df.head()

patient folders: 4
unique studies: 4
unique series: 17
Modalities found: ['CT']
BodyPartExamined: ['MISSING']
Manufacturers: ['NewTom']
Scanner models: ['NTVGiMK4']


,FilePath,PatientFolder,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID,SOPClassUID,PatientID,StudyDate,StudyDescription,Modality,...,InstitutionName,SeriesNumber,KVP,XRayTubeCurrent,ExposureTime,SliceThickness,Rows,Columns,BitsAllocated,NumberOfFrames
0,C:\Users\julie\AI agent courses\dental_imaging...,P1,1.3.76.13.10010.39.32.19.3132.1600414505.4501,1.76.380.18.10.1131014132236753.30299.91.1,1.76.380.18.10.1131014132236753.30299.91.1,1.2.840.10008.5.1.4.1.1.2,00554076,20200918,Tc arcata dent. inferiore,CT,...,MISSING,1.0,110.0,1.0,3600.0,123.0,274.0,410.0,16.0,NaN
1,C:\Users\julie\AI agent courses\dental_imaging...,P1,1.3.76.13.10010.39.32.19.3132.1600414505.4501,1.76.380.18.10.1131014132236753.30299.92.11,1.76.380.18.10.1131014132236753.30299.92.1,1.2.840.10008.5.1.4.1.1.2,00554076,20200918,Tc arcata dent. inferiore,CT,...,MISSING,11.0,110.0,1.0,3600.0,0.3,352.0,370.0,16.0,NaN
2,C:\Users\julie\AI agent courses\dental_imaging...,P1,1.3.76.13.10010.39.32.19.3132.1600414505.4501,1.76.380.18.10.1131014132236.93200918095010952.21,1.76.380.18.10.1131014132236.93200918103254021...,1.2.840.10008.5.1.4.1.1.2,00554076,20200918,Tc arcata dent. inferiore,CT,...,MISSING,21.0,110.0,1.0,3600.0,1.0,170.0,99.0,16.0,NaN
3,C:\Users\julie\AI agent courses\dental_imaging...,P1,1.3.76.13.10010.39.32.19.3132.1600414505.4501,1.76.380.18.10.1131014132236.95200918094934270.31,1.76.380.18.10.1131014132236.95200918103255031...,1.2.840.10008.5.1.4.1.1.2,00554076,20200918,Tc arcata dent. inferiore,CT,...,MISSING,31.0,110.0,1.0,3600.0,1.0,170.0,1053.0,16.0,NaN
4,C:\Users\julie\AI agent courses\dental_imaging...,P1,1.3.76.13.10010.39.32.19.3132.1600414505.4501,1.76.380.18.10.1131014132236.95200918095956770.32,1.76.380.18.10.1131014132236.95200918103256032...,1.2.840.10008.5.1.4.1.1.2,00554076,20200918,Tc arcata dent. inferiore,CT,...,MISSING,32.0,110.0,1.0,3600.0,1.0,170.0,1027.0,16.0,NaN


In [9]:
# ============================================================
# CELL 1 — Imports & Configuration
# ============================================================
import os
import pydicom
import pandas as pd
import numpy as np

# ── Edit this path ──────────────────────────────────────────
DATA_DIR = r"C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy"


# OMOP Image_occurrence maps at the SERIES level, not instance.
# Set True to keep only one DICOM file per series (faster, cleaner output).
ONE_PER_SERIES = True

print(f"Data directory : {DATA_DIR}")
print(f"One-per-series mode: {ONE_PER_SERIES}")

Data directory : C:\Users\julie\AI agent courses\dental_imaging_project_local\DICOM2OMOP\data\ToothFairy
One-per-series mode: True


In [36]:
# ============================================================
# CELL 2 — Tag Extraction
# Gemini captured acquisition params. This adds:
#   • Patient folder name (ToothFairy strips PHI; folder IS the ID)
#   • All OMOP Image_occurrence required UIDs
#   • BodyPartExamined (critical for dental feasibility check)
#   • SOP Class (tells us if it's truly a CT Image Storage object)
#   • Rows/Columns/BitsAllocated (volume geometry)
#   • Series-level deduplication
# ============================================================

metadata_records = []
seen_series = set()   # for ONE_PER_SERIES deduplication
failed_files = []

print("Scanning for DICOM files...")

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if not file.lower().endswith('.dcm'):
            continue

        file_path = os.path.join(root, file)

        try:
            ds = pydicom.dcmread(file_path, stop_before_pixels=True)

            # ── Series deduplication ────────────────────────
            series_uid = str(ds.get('SeriesInstanceUID', 'MISSING'))
            if ONE_PER_SERIES and series_uid in seen_series:
                continue
            seen_series.add(series_uid)

            # ── Patient folder name (ToothFairy uses P1, P2…) ─
            # Walk up from file to find the top-level patient folder
            rel_path  = os.path.relpath(root, DATA_DIR)
            patient_folder = rel_path.split(os.sep)[0]

            # ── PixelSpacing: store row & col separately ────
            pixel_spacing = ds.get('PixelSpacing', None)
            ps_row = float(pixel_spacing[0]) if pixel_spacing else None
            ps_col = float(pixel_spacing[1]) if pixel_spacing else None

            record = {
                # ── Provenance ──────────────────────────────
                "FilePath"              : file_path,
                "PatientFolder"         : patient_folder,   # ToothFairy's de-ID patient key

                # ── OMOP Image_occurrence required fields ───
                # (these 3 UIDs are the backbone of the DICOM hierarchy)
                "StudyInstanceUID"      : str(ds.get('StudyInstanceUID',  'MISSING')),
                "SeriesInstanceUID"     : series_uid,
                "SOPInstanceUID"        : str(ds.get('SOPInstanceUID',    'MISSING')),
                "SOPClassUID"           : str(ds.get('SOPClassUID',       'MISSING')),

                # ── Patient / visit ─────────────────────────
                "PatientID"             : str(ds.get('PatientID',         'MISSING')),
                "StudyDate"             : str(ds.get('StudyDate',         'MISSING')),
                "StudyDescription"      : str(ds.get('StudyDescription',  'MISSING')),
                "SeriesDescription"     : str(ds.get('SeriesDescription', 'MISSING')),
                "SeriesNumber"          : ds.get('SeriesNumber',           None),

                # ── Modality & anatomy (OMOP vocabulary check) ─
                "Modality"              : str(ds.get('Modality',          'MISSING')),
                "BodyPartExamined"      : str(ds.get('BodyPartExamined',  'MISSING')),
                "PatientPosition"       : str(ds.get('PatientPosition',   'MISSING')),

                # ── Scanner identity (NewTom vs i-CAT split) ─
                "Manufacturer"          : str(ds.get('Manufacturer',             'MISSING')),
                "ManufacturerModelName" : str(ds.get('ManufacturerModelName',    'MISSING')),
                "SoftwareVersions"      : str(ds.get('SoftwareVersions',         'MISSING')),
                "InstitutionName"       : str(ds.get('InstitutionName',          'MISSING')),

                # ── Acquisition params (paper-specific) ─────
                "KVP"                   : ds.get('KVP',                None),
                "XRayTubeCurrent"       : ds.get('XRayTubeCurrent',    None),
                "ExposureTime"          : ds.get('ExposureTime',        None),
                "SliceThickness"        : ds.get('SliceThickness',      None),
                "PixelSpacing_Row"      : ps_row,
                "PixelSpacing_Col"      : ps_col,

                # ── Volume geometry ──────────────────────────
                "Rows"                  : ds.get('Rows',         None),
                "Columns"              : ds.get('Columns',       None),
                "BitsAllocated"         : ds.get('BitsAllocated',None),
                "NumberOfFrames"        : ds.get('NumberOfFrames',None),
            }
            metadata_records.append(record)

        except Exception as e:
            failed_files.append({"FilePath": file_path, "Error": str(e)})

df = pd.DataFrame(metadata_records)

print(f"\n✅ Extracted metadata: {len(df)} series records")
print(f"❌ Failed reads     : {len(failed_files)}")
if failed_files:
    print(pd.DataFrame(failed_files))

Scanning for DICOM files...

✅ Extracted metadata: 17 series records
❌ Failed reads     : 0


In [4]:
# ============================================================
# CELL 3 — High-Level Summary
# What Gemini's df.head(10) doesn't show you
# ============================================================

print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"  Patient folders   : {df['PatientFolder'].nunique()}")
print(f"  Unique studies    : {df['StudyInstanceUID'].nunique()}")
print(f"  Unique series     : {df['SeriesInstanceUID'].nunique()}")
print(f"  Modalities found  : {df['Modality'].unique().tolist()}")
print(f"  BodyPartExamined  : {df['BodyPartExamined'].unique().tolist()}")
print(f"  Manufacturers     : {df['Manufacturer'].unique().tolist()}")
print(f"  Scanner models    : {df['ManufacturerModelName'].unique().tolist()}")

# ============================================================
# CELL 4 — Scanner Split (NewTom vs i-CAT)
# The ToothFairy paper describes two distinct acquisition
# protocols. This checks whether both appear in your sample.
# ============================================================

print("SCANNER DISTRIBUTION")
print("-" * 40)
scanner_summary = (
    df.groupby(['Manufacturer', 'ManufacturerModelName'])
    .agg(
        series_count=('SeriesInstanceUID', 'count'),
        patients=('PatientFolder', 'nunique'),
        kvp_mean=('KVP', 'mean'),
        slice_thickness_mean=('SliceThickness', 'mean'),
    )
    .reset_index()
)
print(scanner_summary.to_string(index=False))

DATASET OVERVIEW
  Patient folders   : 4
  Unique studies    : 4
  Unique series     : 17
  Modalities found  : ['CT']
  BodyPartExamined  : ['MISSING']
  Manufacturers     : ['NewTom']
  Scanner models    : ['NTVGiMK4']
SCANNER DISTRIBUTION
----------------------------------------
Manufacturer ManufacturerModelName  series_count  patients  kvp_mean  slice_thickness_mean
      NewTom              NTVGiMK4            17         4     110.0             29.511765


In [35]:
# ============================================================
# CELL 5 — OMOP Feasibility Gap Analysis
# The core deliverable Gemini's code never produces.
# For each field the DICOM2OMOP pipeline needs, what % of
# your records actually have it?
# ============================================================

# Fields that DICOM2OMOP's Image_occurrence table requires
OMOP_REQUIRED = [
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "SOPInstanceUID",
    "SOPClassUID",
    "Modality",
    "BodyPartExamined",
    "StudyDate",
    "PatientID",
    "PatientPosition",
    "Manufacturer",
    "ManufacturerModelName",
    "KVP",
    "SliceThickness",
    "PixelSpacing_Row",
    "Rows",
    "Columns",
]

print("OMOP FIELD COVERAGE (feasibility check)")
print("-" * 45)

gap_rows = []
for col in OMOP_REQUIRED:
    if col not in df.columns:
        pct_present = 0.0
    else:
        # Treat 'MISSING', 'None', empty string as missing
        is_present = df[col].apply(
            lambda x: x not in [None, 'MISSING', 'None', '', 'nan', np.nan]
        )
        pct_present = is_present.mean() * 100
    status = "✅" if pct_present == 100 else ("⚠️" if pct_present > 0 else "❌")
    gap_rows.append({
        "OMOP Field"   : col,
        "% Present"    : f"{pct_present:.1f}%",
        "Status"       : status
    })

gap_df = pd.DataFrame(gap_rows)
print(gap_df.to_string(index=False))

OMOP FIELD COVERAGE (feasibility check)
---------------------------------------------
           OMOP Field % Present Status
     StudyInstanceUID    100.0%      ✅
    SeriesInstanceUID    100.0%      ✅
       SOPInstanceUID    100.0%      ✅
          SOPClassUID    100.0%      ✅
             Modality    100.0%      ✅
     BodyPartExamined      0.0%      ❌
            StudyDate    100.0%      ✅
            PatientID    100.0%      ✅
      PatientPosition    100.0%      ✅
         Manufacturer    100.0%      ✅
ManufacturerModelName    100.0%      ✅
                  KVP    100.0%      ✅
       SliceThickness    100.0%      ✅
     PixelSpacing_Row      0.0%      ❌
                 Rows    100.0%      ✅
              Columns    100.0%      ✅


In [6]:
# ============================================================
# CELL 6 — Vocabulary Check: Will Modality & BodyPart Map?
# DICOM2OMOP has concepts for 79 modality codes and 318 body
# part codes. Check whether ToothFairy's values are in scope.
# ============================================================

# Subset of OMOP vocabulary codes confirmed present in DICOM2OMOP
# (from the JAMIA paper). CBCT scanners typically emit 'CT'.
KNOWN_OMOP_MODALITIES = {
    'CT', 'MR', 'PT', 'US', 'DX', 'CR', 'NM', 'XA', 'RF', 'MG'
}

# Body part codes from DICOM Part 16 that appear in the vocabulary
KNOWN_OMOP_BODY_PARTS = {
    'JAW', 'MANDIBLE', 'MAXILLA', 'TEETH', 'HEAD', 'NECK',
    'SKULL', 'BRAIN', 'CHEST', 'ABDOMEN'  # broader set for reference
}

print("VOCABULARY MAPPING CHECK")
print("-" * 45)

for modality in df['Modality'].unique():
    in_vocab = "✅ in OMOP vocab" if modality in KNOWN_OMOP_MODALITIES else "❌ NOT in known vocab"
    print(f"  Modality '{modality}': {in_vocab}")

print()
for bp in df['BodyPartExamined'].unique():
    in_vocab = "✅ in OMOP vocab" if bp in KNOWN_OMOP_BODY_PARTS else "❓ verify in DICOM2OMOP files"
    print(f"  BodyPart '{bp}': {in_vocab}")

print("""
NOTE: 'CT' is the standard DICOM Modality tag for CBCT scanners.
If BodyPartExamined is 'MISSING', this is a key feasibility finding —
dental CBCT datasets may not consistently populate this tag.
""")

VOCABULARY MAPPING CHECK
---------------------------------------------
  Modality 'CT': ✅ in OMOP vocab

  BodyPart 'MISSING': ❓ verify in DICOM2OMOP files

NOTE: 'CT' is the standard DICOM Modality tag for CBCT scanners.
If BodyPartExamined is 'MISSING', this is a key feasibility finding —
dental CBCT datasets may not consistently populate this tag.



In [7]:
# ============================================================
# CELL 7 — Acquisition Parameter Distributions
# Verifies the paper's stated specs: 110 kV, 3 mA, 0.3 mm
# ============================================================

print("ACQUISITION PARAMETERS")
print("-" * 45)
acq_cols = ['KVP', 'XRayTubeCurrent', 'SliceThickness',
            'PixelSpacing_Row', 'PixelSpacing_Col']

print(df[acq_cols].describe().round(3))

print("\nExpected from ToothFairy paper:")
print("  NewTom : 110 kVp, 3 mA, 0.3 mm voxels")
print("  i-CAT  : 'Extended Field' mode, 0.4 mm voxels")

ACQUISITION PARAMETERS
---------------------------------------------
         KVP  XRayTubeCurrent  SliceThickness  PixelSpacing_Row  \
count   17.0           17.000          17.000              17.0   
mean   110.0            1.941          29.512               0.3   
std      0.0            0.748          53.455               0.0   
min    110.0            1.000           0.300               0.3   
25%    110.0            1.000           0.500               0.3   
50%    110.0            2.000           1.000               0.3   
75%    110.0            2.000           1.000               0.3   
max    110.0            3.000         123.000               0.3   

       PixelSpacing_Col  
count              17.0  
mean                0.3  
std                 0.0  
min                 0.3  
25%                 0.3  
50%                 0.3  
75%                 0.3  
max                 0.3  

Expected from ToothFairy paper:
  NewTom : 110 kVp, 3 mA, 0.3 mm voxels
  i-CAT  : 'Extended

In [ ]:
# # ============================================================
# # CELL 8 — Export for NIDCR Laptop Transfer
# # Saves a clean CSV you can open anywhere — no database needed
# # ============================================================

# output_path = os.path.join(DATA_DIR, "toothfairy_dicom_metadata.csv")
# df.to_csv(output_path, index=False)
# print(f"✅ Saved: {output_path}")
# print(f"   Rows: {len(df)} | Columns: {len(df.columns)}")
# print("\nThis CSV is your portable dataset characterization.")
# print("All downstream OMOP mapping work can run from this file alone.")